In [ ]:
# Cell 1: Enhanced FaceAnalyzer with debugging
import os
import sys
import urllib.request
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Setup project path
proj_root = os.path.abspath(os.path.join('..', 'lib'))
if proj_root not in sys.path:
    sys.path.insert(0, proj_root)

try:
    import onnxruntime as ort
except Exception:
    ort = None

try:
    from insightface.app import FaceAnalysis
except Exception as exc:
    raise ImportError("InsightFace is required for FaceAnalyzer") from exc

from face_detector import FaceDetector, select_image_file

# Constants
PROJECT_ROOT = Path('..').resolve()
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Set environment variables
os.environ.setdefault("INSIGHTFACE_HOME", str(MODELS_DIR))
os.environ.setdefault("ONNX_HOME", str(MODELS_DIR))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(MODELS_DIR / "huggingface"))

class EnhancedFaceAnalyzer:
    """Enhanced FaceAnalyzer with better debugging and improved gender/ethnicity detection."""
    
    _RACE_MODEL_URL = (
        "https://huggingface.co/serengil/deepface_models/resolve/main/fairface_race.onnx?download=1"
    )
    _RACE_LABELS = [
        "White", "Black", "Latino/Hispanic", "East Asian", 
        "Southeast Asian", "Indian", "Middle Eastern",
    ]

    def __init__(self, device: str = "auto", debug: bool = True) -> None:
        self.debug = debug
        self._requested_device = (device or "auto").lower()
        self._ctx_id = self._select_ctx_id(self._requested_device)
        self._providers = self._select_providers(self._ctx_id)
        self._insight = None
        self._race_session = None
        self._race_input_layout = None
        self._race_input_name = None
        self._race_confidence_threshold = 0.1
        
        if self.debug:
            print(f"Initialized with device: {device}, ctx_id: {self._ctx_id}")
            print(f"Available providers: {self._providers}")

    @staticmethod
    def _select_ctx_id(device: str) -> int:
        requested = (device or "auto").lower()
        if requested == "cpu":
            return -1
        if requested in {"gpu", "auto"}:
            # Try PyTorch CUDA
            try:
                import torch
                if torch.cuda.is_available():
                    return 0
            except ImportError:
                pass
            # Try ONNX Runtime CUDA
            if ort is not None:
                try:
                    providers = ort.get_available_providers()
                    if any(p.startswith("CUDA") for p in providers):
                        return 0
                except Exception:
                    pass
        return -1

    @staticmethod
    def _select_providers(ctx_id: int) -> Optional[List[str]]:
        if ort is None:
            return None
        try:
            providers = ort.get_available_providers()
        except Exception:
            return None
        if ctx_id == -1:
            return ["CPUExecutionProvider"] if "CPUExecutionProvider" in providers else None
        if "CUDAExecutionProvider" in providers:
            return ["CUDAExecutionProvider", "CPUExecutionProvider"]
        return ["CPUExecutionProvider"] if "CPUExecutionProvider" in providers else None

    def _load_insight(self, ctx_id: int, providers: Optional[List[str]]) -> FaceAnalysis:
        if self.debug:
            print("Loading InsightFace models...")
        app = FaceAnalysis(
            name="buffalo_l",
            root=str(MODELS_DIR),
            providers=providers,
            allowed_modules=["detection", "recognition", "genderage"],
        )
        app.prepare(ctx_id=ctx_id, det_size=(256, 256))
        if self.debug:
            print("InsightFace models loaded successfully")
        return app

    def _load_race_classifier(self, providers: Optional[List[str]]):
        if ort is None:
            if self.debug:
                print("ONNX Runtime not available for race classification")
            return None
        
        model_path = MODELS_DIR / "demography" / "fairface_race.onnx"
        if self.debug:
            print(f"Looking for FairFace model at: {model_path}")
        
        if not model_path.exists():
            if self.debug:
                print("FairFace model not found, downloading...")
            model_path.parent.mkdir(parents=True, exist_ok=True)
            try:
                urllib.request.urlretrieve(self._RACE_MODEL_URL, model_path)
                if self.debug:
                    print("FairFace model downloaded successfully")
            except Exception as e:
                if self.debug:
                    print(f"Failed to download FairFace model: {e}")
                return None
        
        try:
            session = ort.InferenceSession(str(model_path), providers=providers)
            if self.debug:
                print("FairFace model loaded successfully")
            
            # Cache input metadata
            inp = session.get_inputs()[0]
            self._race_input_name = inp.name
            shape = list(inp.shape)
            
            if self.debug:
                print(f"FairFace input - name: {self._race_input_name}, shape: {shape}")
            
            # Determine layout
            if len(shape) == 4:
                if shape[1] == 3:
                    self._race_input_layout = 'NCHW'
                elif shape[-1] == 3:
                    self._race_input_layout = 'NHWC'
                else:
                    self._race_input_layout = 'NCHW'
            else:
                self._race_input_layout = 'NCHW'
                
            if self.debug:
                print(f"Using input layout: {self._race_input_layout}")
            
            return session
        except Exception as e:
            if self.debug:
                print(f"Failed to load FairFace model: {e}")
            return None

    def _ensure_initialized(self) -> None:
        if self._insight is None:
            self._insight = self._load_insight(self._ctx_id, self._providers)
        if self._race_session is None:
            self._race_session = self._load_race_classifier(self._providers)

    def _normalize_gender(self, raw) -> str:
        """Enhanced gender normalization with better InsightFace compatibility."""
        if raw is None:
            return "Unknown"
        
        # Handle numpy arrays and tensors
        if hasattr(raw, 'item'):
            raw = raw.item()
        
        if self.debug:
            print(f"Gender raw value: {raw}, type: {type(raw)}")
        
        if isinstance(raw, (int, float)):
            # InsightFace typically uses: 0=Female, 1=Male
            # Handle different model variations
            if raw < 0.1:  # Very low values = Female
                return "Female"
            elif raw > 0.9:  # Very high values = Male  
                return "Male"
            else:
                # For ambiguous values, use threshold
                return "Female" if raw < 0.5 else "Male"
        
        if isinstance(raw, str):
            lowered = raw.lower()
            if lowered.startswith("f") or lowered == "0":
                return "Female"
            if lowered.startswith("m") or lowered == "1":
                return "Male"
        
        return "Unknown"

    @staticmethod
    def _imagenet_normalize(image: np.ndarray) -> np.ndarray:
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        return (image - mean) / std

    @staticmethod
    def _half_normalize(image: np.ndarray) -> np.ndarray:
        mean = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        std = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        return (image - mean) / std

    def _prep_race_input(self, face_img: np.ndarray, layout: str, strategy: str) -> np.ndarray:
        # Convert BGR -> RGB, resize to 224
        image = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        
        if strategy == 'imagenet':
            image = self._imagenet_normalize(image)
        else:
            image = self._half_normalize(image)
            
        if layout == 'NCHW':
            image = np.transpose(image, (2, 0, 1))
            
        image = image[np.newaxis, ...].astype(np.float32)
        return image

    @staticmethod
    def _softmax(x: np.ndarray) -> np.ndarray:
        x = x.astype(np.float32)
        x = x - np.max(x, axis=-1, keepdims=True)
        e = np.exp(x)
        return e / np.sum(e, axis=-1, keepdims=True)

    def _predict_race(self, face_img: np.ndarray) -> str:
        if self._race_session is None:
            if self.debug:
                print("Race session not available")
            return "Unknown"
        
        if self._race_input_name is None or self._race_input_layout is None:
            if self.debug:
                print("Race input metadata not available")
            return "Unknown"
            
        try:
            # Try two common normalization strategies
            for strategy in ('imagenet', 'half'):
                if self.debug:
                    print(f"Trying normalization strategy: {strategy}")
                    
                inp = self._prep_race_input(face_img, self._race_input_layout, strategy)
                outputs = self._race_session.run(None, {self._race_input_name: inp})[0]
                
                if outputs.ndim == 2:
                    outputs = outputs[0]
                    
                probs = self._softmax(outputs)
                best_idx = int(np.argmax(probs))
                best_prob = float(probs[best_idx])
                
                if self.debug:
                    print(f"Best index: {best_idx}, probability: {best_prob:.3f}")
                
                if len(self._RACE_LABELS) == probs.shape[0]:
                    label = self._RACE_LABELS[best_idx]
                else:
                    label = f"Class_{best_idx}"
                    
                if best_prob >= self._race_confidence_threshold:
                    if self.debug:
                        print(f"Race prediction: {label} (confidence: {best_prob:.3f})")
                    return label
                else:
                    if self.debug:
                        print(f"Low confidence: {best_prob:.3f} < {self._race_confidence_threshold}")
            
            return "Unknown"
        except Exception as e:
            if self.debug:
                print(f"Race prediction error: {e}")
            return "Unknown"

    def verify_models(self) -> Dict[str, bool]:
        """Check if all required models are loaded properly."""
        self._ensure_initialized()
        
        status = {
            "insightface_loaded": self._insight is not None,
            "race_classifier_loaded": self._race_session is not None,
        }
        
        # Test InsightFace functionality
        if self._insight is not None:
            try:
                dummy_img = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
                faces = self._insight.get(dummy_img)
                status["insightface_functional"] = True
            except Exception:
                status["insightface_functional"] = False
        
        print("Model Status:", status)
        return status

    def analyze_face(self, face_img: np.ndarray) -> Dict[str, str]:
        try:
            self._ensure_initialized()
            faces = self._insight.get(face_img)
            
            if not faces:
                if self.debug:
                    print("No face detected in provided crop")
                raise ValueError("No face detected in provided crop")
            
            face = faces[0]
            
            # Get raw attributes
            age_val = getattr(face, "age", None)
            gender_val = getattr(face, "gender", None)
            
            if self.debug:
                print(f"DEBUG - Raw age: {age_val}, type: {type(age_val)}")
                print(f"DEBUG - Raw gender: {gender_val}, type: {type(gender_val)}")
            
            # Convert age
            if isinstance(age_val, (int, float)):
                age = str(int(round(age_val)))
            elif hasattr(age_val, 'item'):
                age = str(int(round(age_val.item())))
            else:
                age = "Unknown"
            
            # Convert gender
            gender = self._normalize_gender(gender_val)
            
            # Get ethnicity
            ethnicity = self._predict_race(face_img)
            
            result = {
                "age": age,
                "gender": gender,
                "ethnicity": ethnicity,
            }
            
            if self.debug:
                print(f"Final result: {result}")
            
            return result
            
        except Exception as e:
            if self.debug:
                print(f"Analysis error: {e}")
            return {"age": "Unknown", "gender": "Unknown", "ethnicity": "Unknown"}

    def batch_analyze(self, face_images: List[np.ndarray]) -> List[Dict[str, str]]:
        self._ensure_initialized()
        return [self.analyze_face(img) for img in face_images]

In [ ]:
# Cell 2: Initialize and test the enhanced analyzer
print("Initializing Enhanced Face Analyzer...")
analyzer = EnhancedFaceAnalyzer(debug=True)

print("\nVerifying models...")
model_status = analyzer.verify_models()

# Load image
IMAGE_PATH = None
if IMAGE_PATH is None:
    try:
        IMAGE_PATH = select_image_file()
    except Exception:
        pass
        
if not IMAGE_PATH:
    # Use a sample image path or raise error
    raise RuntimeError('No image selected. Set IMAGE_PATH to a local image path.')

print(f"\nLoading image: {IMAGE_PATH}")
img = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Display original image
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()



In [ ]:
# Cell 3: Detect faces and analyze
print("Detecting faces...")
detector = FaceDetector()
faces = detector.process_frame(img)
print(f'Found {len(faces)} faces')

if not faces:
    print("No faces detected!")
else:
    # Analyze each face with enhanced debugging
    print("\nAnalyzing faces...")
    all_results = []
    
    for i, face in enumerate(faces):
        print(f"\n--- Analyzing Face {i+1} ---")
        face_img = face['face_img']
        
        # Analyze with enhanced debug information
        attrs = analyzer.analyze_face(face_img)
        all_results.append(attrs)
        
        print(f"Face {i+1} attributes: {attrs}")
    
    print("\n=== SUMMARY ===")
    for i, result in enumerate(all_results):
        print(f"Face {i+1}: {result}")

In [ ]:
# Cell 4: Display results with annotated thumbnails
if faces:
    print("Creating annotated thumbnails...")
    thumbs = []
    
    for i, (face, attrs) in enumerate(zip(faces, all_results)):
        face_img = face['face_img']
        
        # Prepare display thumbnail (RGB)
        disp = cv2.resize(face_img, (180, 180))
        disp = cv2.cvtColor(disp, cv2.COLOR_BGR2RGB)
        
        # Create informative label
        label = f"{i+1}: {attrs.get('gender', '?')} {attrs.get('age', '?')}y\n{attrs.get('ethnicity', '?')}"
        thumbs.append((disp, label))
    
    # Show grid with labels
    cols = min(4, max(1, len(thumbs)))
    rows = (len(thumbs) + cols - 1) // cols
    
    plt.figure(figsize=(4 * cols, 4 * rows))
    for idx, (img_th, lbl) in enumerate(thumbs):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(img_th)
        plt.title(lbl, fontsize=10)
        plt.axis('off')
    
    plt.suptitle('Enhanced Face Analysis Results', fontsize=16)
    plt.tight_layout()
    plt.show()
    
else:
    print("No faces to display.")

In [ ]:
# Cell 5: Batch analysis demonstration (optional)
if len(faces) > 1:
    print("Demonstrating batch analysis...")
    face_images = [face['face_img'] for face in faces]
    batch_results = analyzer.batch_analyze(face_images)
    
    print("\nBatch Results:")
    for i, result in enumerate(batch_results):
        print(f"  Face {i+1}: {result}")

In [ ]:
# Cell 6: Model information and troubleshooting
print("=== MODEL INFORMATION ===")
print(f"Models directory: {MODELS_DIR}")
print(f"InsightFace home: {os.environ.get('INSIGHTFACE_HOME')}")
print(f"ONNX home: {os.environ.get('ONNX_HOME')}")

# Check for model files
fairface_path = MODELS_DIR / "demography" / "fairface_race.onnx"
print(f"FairFace model exists: {fairface_path.exists()}")

if fairface_path.exists():
    print(f"FairFace model size: {fairface_path.stat().st_size / (1024*1024):.2f} MB")

# Check InsightFace model directory
insightface_dir = MODELS_DIR / "models" / "buffalo_l"
if insightface_dir.exists():
    print(f"InsightFace models found: {len(list(insightface_dir.glob('*.onnx')))} files")
    for model_file in insightface_dir.glob('*.onnx'):
        print(f"  - {model_file.name} ({model_file.stat().st_size / (1024*1024):.2f} MB)")